# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU**.

Do **not** Run all. Every session: **cell 1 → cell 2 → cell 4 (Drive) → one sweep cell**.

Cell 2 runs `git pull` + reinstall **before** any `apertus_eval_prep` import — always rerun it after opening Colab.

Free Colab dies after 1–2 hours **or** when GPU quota is exhausted. Each item is written to Drive as `{run_id}.partial.jsonl`. If the runtime dies at `[520/800]`, the next session resumes at 521.

Do not treat notebook stdout as a result. Only `results/runs/*.json` plus a registry row is a finished cell.

Use `results/registry_paper.jsonl` (not the n=4 smoke `registry.jsonl`).


In [1]:
# Cell 1 — clone (first visit only) or pull latest
import os
if os.path.exists("pyproject.toml") and os.path.exists("src/apertus_eval_prep"):
    print("Already in repo root")
    !git pull --ff-only
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
    !git pull --ff-only
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"
!git log -1 --oneline


Cloning into 'apertus-eval-prep'...
remote: Enumerating objects: 752, done.
remote: Counting objects: 100% (752/752), done.
remote: Compressing objects: 100% (377/377), done.
remote: Total 752 (delta 425), reused 660 (delta 346), pack-reused 0 (from 0)
Receiving objects: 100% (752/752), 2.68 MiB | 12.92 MiB/s, done.
Resolving deltas: 100% (425/425), done.
/content/apertus-eval-prep
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.7/303.7 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.2 MB/s eta 0:00:00
   ━━━━━

In [2]:
# Cell 2 — pull + GPU check (rerun every session before imports)
import os, sys
from pathlib import Path

if Path("pyproject.toml").exists() and Path("src/apertus_eval_prep").exists():
    pass
elif Path("apertus-eval-prep/pyproject.toml").exists():
    os.chdir("apertus-eval-prep")
else:
    raise FileNotFoundError("Run cell 1 first (clone).")

!git pull --ff-only
!pip -q install -e ".[gpu,viz]"
!git log -1 --oneline

# Editable install can cache an old src/ on sys.path — prefer repo src.
_repo_src = str((Path.cwd() / "src").resolve())
if _repo_src not in sys.path:
    sys.path.insert(0, _repo_src)

import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))

def _suppress_quant_fallback():
    import logging
    import warnings
    logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
    logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
    warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
    warnings.filterwarnings("ignore", module=r"bitsandbytes\..*")

try:
    from apertus_eval_prep.backends.hf import suppress_quantization_warnings
except ImportError:
    suppress_quantization_warnings = _suppress_quant_fallback
suppress_quantization_warnings()

if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")


Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for apertus-eval-prep (pyproject.toml) ... done
c242266 (HEAD -> master, origin/master, origin/HEAD) Land Qwen-3B seed 1/2 from Colab zip; matrix now 19/34.
Tesla T4
official slices already on disk


## Persist to Google Drive

Authorize Drive when prompted. Files live in `MyDrive/apertus-eval-prep-paper/` so a runtime reset does not wipe finished cells.

Also download the zip to your Mac as a second copy.

In [3]:
import os, sys
from pathlib import Path
from google.colab import drive, files

def ensure_repo():
    for p in (Path.cwd(), Path("/content/apertus-eval-prep"), Path("apertus-eval-prep")):
        if (p / "pyproject.toml").exists() and (p / "src" / "apertus_eval_prep").exists():
            os.chdir(p)
            src = str((p / "src").resolve())
            if src not in sys.path:
                sys.path.insert(0, src)
            print("cwd:", Path.cwd(), flush=True)
            return p
    raise FileNotFoundError("Repo not found. Run the clone/pip cell first.")

ensure_repo()
try:
    import apertus_eval_prep  # noqa: F401
except ModuleNotFoundError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".[gpu,viz]"])

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/apertus-eval-prep-paper")
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / "runs").mkdir(exist_ok=True)
Path("results/runs").mkdir(parents=True, exist_ok=True)
os.environ["APERTUS_CHECKPOINT_DIR"] = str(DRIVE / "runs")

if (DRIVE / "registry_paper.jsonl").exists():
    !cp -a {DRIVE}/registry_paper.jsonl results/registry_paper.jsonl
    print("restored registry from Drive")
if any((DRIVE / "runs").iterdir()):
    !cp -a {DRIVE}/runs/. results/runs/
    print("restored runs from Drive")
n_partial = len(list(Path("results/runs").glob("*.partial.jsonl")))
print(f"partial checkpoints on disk: {n_partial}", flush=True)

def save_paper():
    import subprocess
    DRIVE.mkdir(parents=True, exist_ok=True)
    (DRIVE / "runs").mkdir(exist_ok=True)
    if Path("results/registry_paper.jsonl").exists():
        subprocess.check_call(["cp", "-a", "results/registry_paper.jsonl", str(DRIVE / "registry_paper.jsonl")])
    if Path("results/runs").exists():
        subprocess.check_call(["bash", "-lc", f"cp -a results/runs/. {DRIVE}/runs/"])
    n = len(list(Path("results/runs").glob("*.json")))
    print(f"saved to Drive ({n} run JSON files):", DRIVE, flush=True)
    subprocess.check_call(["zip", "-r", "/tmp/paper_matrix_partial.zip", "results/runs", "results/registry_paper.jsonl"])
    files.download("/tmp/paper_matrix_partial.zip")

def sweep(*extra):
    """Run in-process so Colab shows [1/800] live. subprocess.check_call hid all logs."""
    ensure_repo()
    only_model = only_factor = None
    args = list(extra)
    i = 0
    while i < len(args):
        if args[i] == "--only-model" and i + 1 < len(args):
            only_model = args[i + 1]
            i += 2
        elif args[i] == "--only-factor" and i + 1 < len(args):
            only_factor = args[i + 1]
            i += 2
        else:
            raise ValueError(f"unknown sweep arg {args[i]!r}")
    try:
        from apertus_eval_prep.backends.hf import suppress_quantization_warnings
    except ImportError:
        import logging, warnings
        def suppress_quantization_warnings():
            logging.getLogger("bitsandbytes").setLevel(logging.ERROR)
            logging.getLogger("bitsandbytes.autograd._functions").setLevel(logging.ERROR)
            warnings.filterwarnings("ignore", message=".*MatMul8bitLt.*")
    from apertus_eval_prep.sweep import execute_sweep
    suppress_quantization_warnings()
    print(f"sweep in-process model={only_model} factor={only_factor}", flush=True)
    planned = execute_sweep(
        study_path=Path("configs/experiments/stability.yaml"),
        repo_root=Path(".").resolve(),
        out_dir=Path("results/runs"),
        registry_path=Path("results/registry_paper.jsonl"),
        profile="t4",
        only_model=only_model,
        only_factor=only_factor,
    )
    n_skip = sum(1 for p in planned if p["skipped"])
    print({"n_cells": len(planned), "n_skip": n_skip}, flush=True)
    save_paper()

!python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml --profile t4 --dry-run --out-dir results/runs --registry results/registry_paper.jsonl | head -n 50


cwd: /content/apertus-eval-prep
Mounted at /content/drive
restored registry from Drive
restored runs from Drive
partial checkpoints on disk: 3
skip SmolLM2-1.7B-Instruct_control_control_24ffe98d9250761d factor=control=control
skip Qwen2.5-3B-Instruct_control_control_cff017903a47abb9 factor=control=control
skip Phi-3.5-mini-instruct_control_control_31791224954ba45c factor=control=control
skip SmolLM2-1.7B-Instruct_prompt_id_concise_a0852ca6fc3e5c08 factor=prompt_id=concise
skip Qwen2.5-3B-Instruct_prompt_id_concise_e4980863069a102c factor=prompt_id=concise
skip Phi-3.5-mini-instruct_prompt_id_concise_37963fc49f7e8eca factor=prompt_id=concise
skip SmolLM2-1.7B-Instruct_prompt_id_5shot_b6968af4b73708f7 factor=prompt_id=5shot
skip Qwen2.5-3B-Instruct_prompt_id_5shot_8b703d7cb8d9627a factor=prompt_id=5shot
skip Phi-3.5-mini-instruct_prompt_id_5shot_e8c0aa94458abbd2 factor=prompt_id=5shot
run  SmolLM2-1.7B-Instruct_seed_1_4ae7b5af354b3423 factor=seed=1
run  Qwen2.5-3B-Instruct_seed_1_e8b596c

## Session cells (run one per Colab window)

**On GitHub (`registry_paper.jsonl`, 19/34).** Done cells should print `skip`. Left: 15 cells.

| Status | Cell | Job |
|---|---|---|
| **DONE** | 6–12, 15 | controls, all `prompt_id`, SmolLM2 quant, Qwen-7B int4 |
| **DONE** | backend nb | SmolLM2 + Qwen-3B + Phi `backend=vllm` |
| **DONE** | 17–18 | SmolLM2 + Qwen-3B **`seed`** (both match control) |
| **RUN NOW** | 13 | Qwen-3B **`quantization`** |
| Next | 14, 19 | Phi quant; Phi seed |
| Later | — | `sampled` T=0.7 |
| Inventory | — | [`paper/run_status.md`](../paper/run_status.md) |

Order: **cell 1 → 2 → 4 → one sweep**. Resume from `.partial.jsonl` after pull.


In [ ]:
# DONE on GitHub — skip. Only run if dry-run says run not skip.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — Qwen 3B control (515/800). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — Phi-3.5 control (536/800). Skip unless dry-run says run.
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "control")

In [ ]:
# DONE on GitHub — SmolLM2 prompt_id (concise 186/800, 5shot 274/800). Skip unless dry-run says run.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "prompt_id")

In [ ]:
# DONE on GitHub — Qwen-3B prompt_id (concise 410/800, 5shot 549/800). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "prompt_id")


In [ ]:
# DONE on GitHub — Phi prompt_id (concise 471/800, 5shot 451/800). Skip unless dry-run says run.
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "prompt_id")


In [ ]:
# DONE on GitHub — SmolLM2 quantization (int8 334/800, int4 309/800). Skip unless dry-run says run.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "quantization")


In [ ]:
# RUN NOW — Qwen-3B quantization only (int8 + int4). Two 800-item runs.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "quantization")


In [ ]:
# Phi quantization (int8 + int4). If bitsandbytes CPU-offload error: Runtime → Restart session, then cell 1→2→4→this cell (resume keeps .partial.jsonl).
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "quantization")


cwd: /content/apertus-eval-prep
sweep in-process model=microsoft/Phi-3.5-mini-instruct factor=quantization
run  Phi-3.5-mini-instruct_quantization_int8_cfd71ce6317d4a54 factor=quantization=int8
resume 494/800 from Phi-3.5-mini-instruct_quantization_int8_cfd71ce6317d4a54.partial.jsonl


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

[495/800] hellaswag/eval/5907 correct=True pred='D' ttft_ms=1692.15
[496/800] hellaswag/eval/0321 correct=True pred='B' ttft_ms=147.55
[497/800] hellaswag/eval/9030 correct=False pred='C' ttft_ms=234.25
[498/800] hellaswag/eval/7690 correct=True pred='C' ttft_ms=181.26
[499/800] hellaswag/eval/5748 correct=True pred='B' ttft_ms=234.76
[500/800] hellaswag/eval/9814 correct=True pred='C' ttft_ms=172.31
[501/800] hellaswag/eval/6428 correct=True pred='D' ttft_ms=168.26
[502/800] hellaswag/eval/3645 correct=True pred='A' ttft_ms=163.11
[503/800] hellaswag/eval/5061 correct=True pred='A' ttft_ms=182.28
[504/800] hellaswag/eval/4963 correct=True pred='B' ttft_ms=173.49
[505/800] hellaswag/eval/5780 correct=True pred='B' ttft_ms=191.07
[506/800] hellaswag/eval/9035 correct=True pred='D' ttft_ms=157.98
[507/800] hellaswag/eval/2619 correct=True pred='B' ttft_ms=172.44
[508/800] hellaswag/eval/1416 correct=True pred='C' ttft_ms=172.43
[509/800] hellaswag/eval/4937 correct=False pred='D' ttft_ms

In [ ]:
# DONE on GitHub — Qwen-7B int4 (543/800). T4 skips 7B fp16 / int8 / vLLM.
sweep("--only-model", "Qwen/Qwen2.5-7B-Instruct")


## Seed factor (greedy seeds 1 and 2)

Control already used `seed=0`. These cells run **seed 1 and seed 2** only (two × 800 per model).

Greedy decode often makes seed a weak / no-op factor — still fill the matrix cells. One model per Colab session.


In [ ]:
# DONE on GitHub — SmolLM2 seed 1+2 (318/800 each = control). Skip unless dry-run says run.
sweep("--only-model", "HuggingFaceTB/SmolLM2-1.7B-Instruct", "--only-factor", "seed")


In [ ]:
# DONE on GitHub — Qwen-3B seed 1+2 (515/800 both = control). Skip unless dry-run says run.
sweep("--only-model", "Qwen/Qwen2.5-3B-Instruct", "--only-factor", "seed")


In [5]:
# NEXT — Phi-3.5 seed 1 + 2 (two 800-item runs).
sweep("--only-model", "microsoft/Phi-3.5-mini-instruct", "--only-factor", "seed")


cwd: /content/apertus-eval-prep
sweep in-process model=microsoft/Phi-3.5-mini-instruct factor=seed
run  Phi-3.5-mini-instruct_seed_1_f6611690d3503f3e factor=seed=1


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Report (only after several cells exist)

cwd must be `/content/apertus-eval-prep`. Do not run this instead of a sweep.

In [ ]:
from google.colab import files
from pathlib import Path

assert Path("results/registry_paper.jsonl").exists(), "No paper registry in this runtime. Restore from Drive first."
!python -m apertus_eval_prep report --registry results/registry_paper.jsonl --out reports/stability_paper
!python -m apertus_eval_prep paper-tables --registry results/registry_paper.jsonl --out paper/_generated_tables.md
!zip -r paper_matrix_artifacts.zip results/runs results/registry_paper.jsonl reports/stability_paper paper/_generated_tables.md
print("zip bytes", Path("paper_matrix_artifacts.zip").stat().st_size)
files.download("paper_matrix_artifacts.zip")

Unpack the zip (or Drive folder) into the Mac clone. Commit `results/registry_paper.jsonl` and new `results/runs/*.json`. Do not edit numbers.